# 06 - Launch the v2 prune/eval sweep (30 jobs)

Orchestrates the full v2 grid from the experiment spec: **2 pruners
x 5 calibration domains x 3 seeds = 30 EC2 spot GPU jobs**, each
running ``infra/runners/run_prune_eval_sweep.py`` (registered as
the ``prune_eval_sweep`` runner) over base model
``Qwen/Qwen2-7B``. Every job sweeps pruning levels 10..80 (step
10); exactly one job (the first launched) also includes level 0
so there is a single shared, unpruned baseline.

Each job prunes with one pruner (`wanda` or `sparsegpt`) using
calibration texts drawn from one domain
(`math` / `mathqa` / `coding` / `mbpp` / `mcq`) and one seed
(0, 1, 2 -- used as the calibration chunk index, see
`CALIBRATION_SEED` in the runner contract), then evaluates
teacher-forced log-probabilities on **all five** domains at every
level.

Cells:

1. bootstrap
2. grid definition (plain data) + per-domain calibration chunk
   size, computed from each task adapter's train split
3. spot capacity probe (separate instance-type lists per pruner)
4. launch loop over the 30 jobs, resumable via
   `experiment_config_v2.json`
5. monitoring: per-run S3 progress table
6. **STOP-ALL** -- manual-only, terminates recorded instances

Prerequisite: `01_setup_aws.ipynb` (S3 bucket + instance profile)
and package P5's `prune_eval_sweep` runner must be present under
`infra/runners/run_prune_eval_sweep.py` and registered in
`infra/provisioning/launch_gpu_instance.py::RUNNER_RELPATHS`.

**This notebook only launches jobs when cell 4 is run.** Cells 1-3
and 5 are read-only / informational and safe to re-run at any
time; cell 6 is destructive and gated behind an explicit flag.


In [ ]:
import os
import sys
from pathlib import Path

# Locate repo root regardless of where the notebook is opened from.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))
print("AWS_REGION  =", os.environ.get("AWS_REGION"))


## Grid definition

The full v2 grid (see spec section C5):

* **Domains** (calibration + eval specs, identical 5-benchmark
  set on both sides): `math` (GSM8K), `mathqa` (MathQA),
  `coding` (HumanEval+), `mbpp` (MBPP), `mcq` (ARC-Challenge).
* **Pruners**: `wanda`, `sparsegpt`.
* **Seeds**: `0, 1, 2` -- doubles as the `CALIBRATION_SEED` chunk
  index `k` the runner uses to slice
  `train_split[k*C : (k+1)*C]` for calibration texts.
* **Levels**: `10, 20, ..., 80`; the first job launched also gets
  level `0` (the shared baseline).

> **NOTE (package boundary):** `MATHQA_SPEC` / `MBPP_SPEC` below
> assume task-adapter registrations that package P4 owns
> (`mathqa`: a new `TaskAdapter` subclass parsing MathQA's
> single-string options field; `mbpp`: `CodingTaskAdapter` pointed
> at `evalplus/mbppplus`, which already fits the existing
> `coding[:<dataset_name>[:<test_split>]]` spec grammar). P4
> reports the exact strings in its own final message; if P4 chose
> different spec strings, update the two constants below -- no
> other cell needs to change.

Chunk size per domain is `min(128, len(train_split) // 3)`,
computed here (not hard-coded) from each adapter's actual train
split so the grid stays correct if a dataset's size changes
upstream. Per spec this must fail loudly if a domain's train
split is too small (`< 16`) to support 3 non-overlapping chunks
of at least that size.


In [ ]:
import json
import math as _math

from pruning_metrics.evals.tasks.registry import build_adapter_from_spec

BASE_MODEL_ID = "Qwen/Qwen2-7B"

SPLIT_SEED = 65320
TRAIN_FRAC = 0.8
MAX_CALIBRATION_TOKENS = 2048
TF_TOP_K = 10
NUM_TF_SAMPLES = 200
TF_SEED = 65320

PRUNERS = ["wanda", "sparsegpt"]
SEEDS = [0, 1, 2]
PRUNING_LEVELS = [10, 20, 30, 40, 50, 60, 70, 80]

# Domain specs: (label, adapter spec string). Eval uses the same
# 5 specs as calibration (spec C5: "Eval benchmarks: the same 5
# specs"), so one list serves both roles.
# NOTE: the Hub now requires namespaced dataset ids ("openai/gsm8k", not
# "gsm8k") -- the bare id crashes on current huggingface_hub versions.
MATH_SPEC = "math:openai/gsm8k:main"
MATHQA_SPEC = "mathqa:allenai/math_qa"
CODING_SPEC = "coding:evalplus/humanevalplus:test"
MBPP_SPEC = "mbpp:evalplus/mbppplus:test"  # P4's dedicated MBPP+ adapter (schema differs from HumanEval+)
MCQ_SPEC = "mcq:allenai/ai2_arc:ARC-Challenge"

DOMAINS = [
    ("math", MATH_SPEC),
    ("mathqa", MATHQA_SPEC),
    ("coding", CODING_SPEC),
    ("mbpp", MBPP_SPEC),
    ("mcq", MCQ_SPEC),
]
EVAL_DATASET_SPECS = [spec for _label, spec in DOMAINS]

# Instance priority + shared spot bid ceiling, per pruner (C5).
SPARSEGPT_INSTANCE_TYPES = ["g6e.xlarge", "g6e.2xlarge"]
WANDA_INSTANCE_TYPES = ["g5.2xlarge", "g6.xlarge", "g6e.xlarge"]
MAX_SPOT_PRICE = 2.50
REGION_PRIORITY = ["us-east-1", "us-west-2", "us-east-2"]

MIN_TRAIN_RECORDS_FOR_CHUNKING = 16  # fail below this (C5)
MAX_CHUNK_SIZE = 128


def compute_chunk_size(spec: str) -> int:
    """Return ``min(128, len(train_split) // 3)`` for a domain spec.

    Parameters
    ----------
    spec:
        Task-adapter spec string (``registry.build_adapter_from_spec``).

    Returns
    -------
    int
        Calibration chunk size shared by all 3 seeds of this domain.

    Raises
    ------
    ValueError
        If the resulting chunk size is below
        ``MIN_TRAIN_RECORDS_FOR_CHUNKING`` (spec C5: "fail if < 16").
    """

    adapter = build_adapter_from_spec(spec)
    train_records, _test_records = adapter.train_test_split(
        seed=SPLIT_SEED, train_frac=TRAIN_FRAC
    )
    chunk_size = min(MAX_CHUNK_SIZE, len(train_records) // 3)
    if chunk_size < MIN_TRAIN_RECORDS_FOR_CHUNKING:
        raise ValueError(
            f"Domain {spec!r} train split too small for chunking: "
            f"{len(train_records)} records -> chunk_size={chunk_size} "
            f"< {MIN_TRAIN_RECORDS_FOR_CHUNKING}."
        )
    return chunk_size


# Design: compute (rather than hard-code) chunk sizes so the grid
# self-corrects if an upstream dataset's row count changes; each
# domain is independent so one missing/broken adapter doesn't
# block launching jobs for the others -- we record `None` and
# print a NOTE instead of raising, matching the "tolerant of
# partial state" convention used by notebooks 04/05.
CHUNK_SIZES: dict[str, int | None] = {}
for _label, _spec in DOMAINS:
    try:
        CHUNK_SIZES[_label] = compute_chunk_size(_spec)
    except Exception as exc:  # pylint: disable=broad-exception-caught
        CHUNK_SIZES[_label] = None
        print(f"NOTE: chunk size for domain {_label!r} ({_spec!r}) "
              f"unavailable: {exc!r}")

print(json.dumps({
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "PRUNERS": PRUNERS,
    "SEEDS": SEEDS,
    "PRUNING_LEVELS": PRUNING_LEVELS,
    "DOMAINS": DOMAINS,
    "CHUNK_SIZES": CHUNK_SIZES,
}, indent=2))


# Shared launch-record state (used by the launch, monitor, and STOP-ALL
# cells; defined here so those cells work right after a kernel restart).
EXPERIMENT_CONFIG_PATH = Path(os.getcwd()) / "experiment_config_v2.json"
RESULTS_PREFIX = "prune_eval_v2"


def _load_experiment_config() -> dict:
    if EXPERIMENT_CONFIG_PATH.exists():
        return json.loads(EXPERIMENT_CONFIG_PATH.read_text(encoding="utf-8"))
    return {"jobs": []}


def _save_experiment_config(cfg: dict) -> None:
    EXPERIMENT_CONFIG_PATH.write_text(
        json.dumps(cfg, indent=2), encoding="utf-8"
    )


def _job_key(pruner: str, domain: str, seed: int) -> str:
    return f"{pruner}__{domain}__seed{seed}"


## Find spot capacity

Probes spot capacity once per pruner (each pruner has its own
instance-type priority list, C5). `launch_runner_with_fallback`
re-probes automatically if every candidate here is exhausted at
launch time, so this cell just needs a reasonable starting point.


In [ ]:
import os

from pruning_metrics.notebook_helpers import find_capacity

AWS_PROFILE = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get(
    "RESULTS_BUCKET", "pruning-metrics-results-414266451290"
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

CANDIDATES_BY_PRUNER = {}
for _pruner, _instance_types in (
    ("sparsegpt", SPARSEGPT_INSTANCE_TYPES),
    ("wanda", WANDA_INSTANCE_TYPES),
):
    _candidates = find_capacity(
        regions=tuple(REGION_PRIORITY),
        instance_types=tuple(_instance_types),
        aws_profile=AWS_PROFILE,
    )
    assert _candidates, f"No spot capacity found for pruner={_pruner!r}."
    CANDIDATES_BY_PRUNER[_pruner] = _candidates
    _top = _candidates[0]
    print(
        f"[{_pruner}] top candidate: {_top['region']} "
        f"{_top['availability_zone']} {_top['instance_type']} "
        f"@ ${_top['spot_price_usd_per_hour']:.4f}/h "
        f"({len(_candidates)} candidates total)"
    )


## Launch the 30 jobs

One job per `(pruner, domain, seed)`. Launches are staggered by
20 s (EC2 `RunInstances` rate limits) and recorded incrementally
into `experiment_config_v2.json` right after each launch, so a
crashed/restarted kernel resumes from wherever it left off
(already-recorded `(pruner, domain, seed)` combos are skipped).

Per spec C5, exactly the first job launched (in iteration order)
also includes pruning level `0` -- the single shared baseline
evaluated once, not once per job.

Launches run in **quota-aware waves**: the account's G-family spot
quota is 64 vCPUs, so the cell waits (polling every 2 min) whenever
launching the next box would exceed a 56-vCPU budget. Finished runners
self-terminate and free quota, so the cell eventually drains the whole
grid; expect the full launch to take a few hours of wall clock.


In [ ]:
import time

import boto3

from pruning_metrics.notebook_helpers import (
    InsufficientCapacityError,
    QuotaExhaustedError,
    launch_runner_with_fallback,
    render_run_id_default,
)

LAUNCH_STAGGER_SECONDS = 20.0

# The account's "All G and VT Spot Instance Requests" quota is 64 vCPUs
# (us-east-1, checked 2026-07-19). Launch in waves that stay under it;
# runners self-terminate, so finished jobs free quota automatically.
VCPUS_BY_TYPE = {
    "g6e.xlarge": 4,
    "g6e.2xlarge": 8,
    "g5.2xlarge": 8,
    "g6.xlarge": 4,
}
SPOT_VCPU_BUDGET = 56  # headroom under the 64-vCPU quota
QUOTA_POLL_SECONDS = 120.0

# Every domain must have resolved a chunk size before launching: eval specs
# include all 5 domains, so a missing calibration domain would silently
# unbalance the grid (calibrations x evals mismatch).
_missing = [label for label, size in CHUNK_SIZES.items() if size is None]
assert not _missing, (
    f"Chunk size unavailable for domain(s) {_missing}; fix the adapter/spec "
    "(see NOTEs printed by the grid-definition cell) before launching."
)


def _active_spot_vcpus(cfg: dict) -> int:
    """Sum vCPUs of recorded instances still pending/running, per region."""

    by_region: dict[str, list[str]] = {}
    for job in cfg["jobs"]:
        if job.get("instance_id"):
            by_region.setdefault(job["region"], []).append(job["instance_id"])
    total = 0
    for region, ids in by_region.items():
        ec2 = boto3.session.Session(
            profile_name=AWS_PROFILE
        ).client("ec2", region_name=region)
        paginator = ec2.get_paginator("describe_instances")
        for page in paginator.paginate(
            Filters=[{"Name": "instance-id", "Values": ids}]
        ):
            for reservation in page.get("Reservations", []):
                for inst in reservation.get("Instances", []):
                    if inst["State"]["Name"] in ("pending", "running"):
                        total += VCPUS_BY_TYPE.get(inst["InstanceType"], 8)
    return total


experiment_config = _load_experiment_config()
already_launched = {
    _job_key(job["pruner"], job["domain"], job["seed"])
    for job in experiment_config["jobs"]
}
# Baseline is a property of the FIRST job across the whole grid,
# not the first job of this process -- so it must be tracked in
# the persisted config (not a local variable) to survive a
# kernel restart mid-grid.
baseline_already_assigned = any(
    0 in job["levels"] for job in experiment_config["jobs"]
)

print(f"{len(already_launched)} job(s) already recorded; resuming.")

for pruner in PRUNERS:
    instance_types = (
        SPARSEGPT_INSTANCE_TYPES if pruner == "sparsegpt" else WANDA_INSTANCE_TYPES
    )
    # Budget guardrail: never bid above MAX_SPOT_PRICE regardless of what
    # the capacity probe suggests.
    candidates = [
        c
        for c in CANDIDATES_BY_PRUNER[pruner]
        if float(c["max_bid_usd_per_hour"]) <= MAX_SPOT_PRICE
    ]
    assert candidates, (
        f"No {pruner} capacity candidates under MAX_SPOT_PRICE="
        f"{MAX_SPOT_PRICE}; re-run the capacity cell or raise the cap."
    )
    for domain_label, domain_spec in DOMAINS:
        chunk_size = CHUNK_SIZES[domain_label]
        for seed in SEEDS:
            key = _job_key(pruner, domain_label, seed)
            if key in already_launched:
                print(f"SKIP {key}: already recorded.")
                continue

            # Quota gate: wait until the next box fits under the budget.
            need = max(VCPUS_BY_TYPE.get(t, 8) for t in instance_types)
            while True:
                active = _active_spot_vcpus(experiment_config)
                if active + need <= SPOT_VCPU_BUDGET:
                    break
                print(
                    f"quota gate: {active} spot vCPUs active; waiting "
                    f"{QUOTA_POLL_SECONDS:.0f}s before launching {key} ..."
                )
                time.sleep(QUOTA_POLL_SECONDS)

            levels = list(PRUNING_LEVELS)
            is_baseline_job = not baseline_already_assigned
            if is_baseline_job:
                levels = [0] + levels

            runner_env = {
                "BASE_MODEL_ID": BASE_MODEL_ID,
                "PRUNER": pruner,
                "CALIBRATION_DATASET_SPEC": domain_spec,
                "SPLIT_SEED": SPLIT_SEED,
                "TRAIN_FRAC": TRAIN_FRAC,
                "CALIBRATION_SEED": seed,
                "CALIBRATION_CHUNK_SIZE": chunk_size,
                "MAX_CALIBRATION_TOKENS": MAX_CALIBRATION_TOKENS,
                "PRUNING_LEVELS": ",".join(str(lv) for lv in levels),
                "EVAL_DATASET_SPECS": ",".join(EVAL_DATASET_SPECS),
                "TF_TOP_K": TF_TOP_K,
                "NUM_TF_SAMPLES": NUM_TF_SAMPLES,
                "TF_SEED": TF_SEED,
            }
            run_id = render_run_id_default()
            launched = None
            for attempt in (1, 2):
                try:
                    launched = launch_runner_with_fallback(
                        candidates,
                        runner="prune_eval_sweep",
                        runner_env=runner_env,
                        results_bucket=RESULTS_BUCKET,
                        results_prefix=RESULTS_PREFIX,
                        run_id=run_id,
                        aws_profile=AWS_PROFILE,
                        hf_token=HF_TOKEN,
                        name_tag=f"pm-v2-{pruner}-{domain_label}-s{seed}",
                        recheck_regions=REGION_PRIORITY,
                        recheck_instance_types=instance_types,
                    )
                    break
                except (InsufficientCapacityError, QuotaExhaustedError) as exc:
                    print(f"NOTE {key} attempt {attempt}: {exc!r}")
                    if attempt == 1:
                        time.sleep(300.0)
            if launched is None:
                print(f"SKIP {key}: no capacity after retry -- re-run this "
                      "cell later to resume the remaining jobs.")
                continue

            job_record = {
                "pruner": pruner,
                "domain": domain_label,
                "domain_spec": domain_spec,
                "seed": seed,
                "chunk_size": chunk_size,
                "levels": levels,
                "is_baseline": is_baseline_job,
                "run_id": launched.run_id,
                "instance_id": launched.instance_id,
                "region": launched.region,
                "availability_zone": launched.availability_zone,
                "instance_type": launched.instance_type,
                "uri": launched.results_uri,
            }
            experiment_config["jobs"].append(job_record)
            # Persist after every launch (not just at the end) so a
            # crash mid-grid loses at most the in-flight launch.
            _save_experiment_config(experiment_config)
            already_launched.add(key)
            if is_baseline_job:
                baseline_already_assigned = True

            print(
                f"[{key}] {launched.instance_id} run_id={launched.run_id} "
                f"az={launched.availability_zone} type={launched.instance_type} "
                f"levels={levels} -> {launched.results_uri}"
            )
            time.sleep(LAUNCH_STAGGER_SECONDS)

print(f"\n{len(experiment_config['jobs'])} / 30 jobs recorded in "
      f"{EXPERIMENT_CONFIG_PATH}")


## Monitor progress

For every recorded job, lists its S3 prefix and counts how many
distinct `level=NN` sub-prefixes have appeared (each level is
uploaded once its masks + all `EVAL_DATASET_SPECS` teacher-forced
outputs are written, per the runner's per-`(level, bench)` sync
convention). Safe to re-run at any time; read-only.


In [ ]:
import re

from pruning_metrics.notebook_helpers import list_results

LEVEL_PREFIX_RE = re.compile(r"level=(\d+)/")

experiment_config = _load_experiment_config()
total_levels_target = len(PRUNING_LEVELS)  # +1 for the baseline job only

print(
    f"{'pruner':<10} {'domain':<8} {'seed':<5} {'levels_done':<12} "
    f"{'run_id':<28} instance_id"
)
for job in experiment_config["jobs"]:
    prefix = f"{RESULTS_PREFIX}/{job['run_id']}/"
    entries = list_results(RESULTS_BUCKET, prefix, aws_profile=AWS_PROFILE)
    levels_seen = {
        m.group(1)
        for entry in entries
        for m in [LEVEL_PREFIX_RE.search(entry["key"])]
        if m
    }
    target = total_levels_target + (1 if job.get("is_baseline") else 0)
    print(
        f"{job['pruner']:<10} {job['domain']:<8} {job['seed']:<5} "
        f"{len(levels_seen)}/{target:<10} {job['run_id']:<28} "
        f"{job['instance_id']}"
    )


## STOP-ALL -- manual only, DESTRUCTIVE

Terminates every EC2 instance recorded in
`experiment_config_v2.json`. This does **not** run automatically:
set `CONFIRM_TERMINATE_ALL_JOBS = True` in the cell below and
re-run it yourself when you actually want to tear the sweep down
(e.g. after the grid finishes, or to abandon a bad run). Leaving
it `False` (the default) makes this cell a no-op, so the notebook
as a whole remains safe to execute top-to-bottom.


In [ ]:
import boto3

# DESTRUCTIVE: flip to True and re-run this cell to terminate
# every recorded instance_id. Left False so the notebook can be
# executed top-to-bottom without accidentally tearing down a
# live sweep.
CONFIRM_TERMINATE_ALL_JOBS = False

if not CONFIRM_TERMINATE_ALL_JOBS:
    print("CONFIRM_TERMINATE_ALL_JOBS is False; not terminating anything.")
else:
    experiment_config = _load_experiment_config()
    by_region: dict[str, list[str]] = {}
    for job in experiment_config["jobs"]:
        if job["instance_id"]:
            by_region.setdefault(job["region"], []).append(job["instance_id"])

    session = boto3.session.Session(profile_name=AWS_PROFILE)
    for region, instance_ids in by_region.items():
        ec2 = session.client("ec2", region_name=region)
        response = ec2.terminate_instances(InstanceIds=instance_ids)
        for item in response.get("TerminatingInstances", []):
            print(
                f"[{region}] {item['InstanceId']}: "
                f"{item['PreviousState']['Name']} -> {item['CurrentState']['Name']}"
            )
